# Steam Dataset Download

Purpose: This notebook downloads the Steam data used for the DSAN 6600 final project from Hugging Face.

**Dataset:** `frankjc2022/steam-dataset`

This project uses:
- `user_games` — user-game interaction data
- `games` — Steam game metadata

The downloaded files are saved as Parquet under `data/raw/`.

## 1. Import Packages

In [1]:
import sys

print(sys.executable)
print(sys.version)

/opt/anaconda3/bin/python
3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]


In [2]:
#Use Requirements for packages if needed
#%pip install -r ../requirements.txt
%pip install ipykernel
from pathlib import Path
from datetime import datetime, timezone
import json
from datasets import load_dataset

Note: you may need to restart the kernel to use updated packages.


## 2. Project paths and dataset configuration

This notebook assumes it is stored in the repository's `notebooks/` folder.

In [3]:
DATASET_ID = "frankjc2022/steam-dataset"
CONFIGS = ["user_games", "games"]

#Set file path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)

Project root: /Users/robertgeorge/Desktop/DSAN Files/6600/dsan6600-finalproject
Raw data directory: /Users/robertgeorge/Desktop/DSAN Files/6600/dsan6600-finalproject/data/raw


## 3. Download helper

If a local Parquet file already exists, this function skips the download by default.

In [4]:
def download_config(config_name, force=False):
    output_path = RAW_DATA_DIR / f"{config_name}.parquet"

    if output_path.exists() and not force:
        print(f"[SKIP] {output_path.name} already exists.")
        return {
            "config": config_name,
            "output": str(output_path.relative_to(PROJECT_ROOT)),
            "status": "existing_file",
        }

    print(f"Downloading '{config_name}' from {DATASET_ID} ...")

    dataset = load_dataset(
        DATASET_ID,
        name=config_name,
        split="train",
    )

    print(f"Rows:    {dataset.num_rows:,}")
    print(f"Columns: {dataset.num_columns}")
    print(f"Fields:  {dataset.column_names}")

    dataset.to_parquet(str(output_path))

    size_mb = output_path.stat().st_size / (1024 ** 2)
    print(f"Saved {output_path.name} ({size_mb:,.1f} MB)")

    return {
        "config": config_name,
        "rows": dataset.num_rows,
        "columns": dataset.num_columns,
        "column_names": dataset.column_names,
        "output": str(output_path.relative_to(PROJECT_ROOT)),
        "size_mb": round(size_mb, 2),
        "status": "downloaded",
    }


## 4. Download the project datasets

In [5]:
results = []

for config_name in CONFIGS:
    result = download_config(config_name)
    results.append(result)
    print()

results


README.md: 0.00B [00:00, ?B/s]

data/v1/user_games.parquet:   0%|          | 0.00/506M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11285290 [00:00<?, ? examples/s]

Rows:    11,285,290
Columns: 9
Fields:  ['steamid', 'appid', 'game_count', 'game_name', 'playtime_forever', 'img_icon_url', 'has_community_visible_stats', 'has_leaderboards', 'content_descriptorids']


Creating parquet from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

...

Generating train split:   0%|          | 0/47811 [00:00<?, ? examples/s]

Rows:    47,811
Columns: 29
Fields:  ['appid', 'type', 'name', 'is_free', 'required_age', 'short_description', 'detailed_description', 'about_the_game', 'supported_languages', 'developers', 'publishers', 'platforms', 'categories', 'genres', 'pc_requirements', 'mac_requirements', 'linux_requirements', 'price_currency', 'price_initial', 'price_final', 'price_discount_percent', 'price_final_formatted', 'metacritic_score', 'metacritic_url', 'recommendations_total', 'release_date', 'release_coming_soon', 'content_descriptor_ids', 'content_descriptor_notes']


Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Saved games.parquet (133.2 MB)



[{'config': 'user_games',
  'rows': 11285290,
  'columns': 9,
  'column_names': ['steamid',
   'appid',
   'game_count',
   'game_name',
   'playtime_forever',
   'img_icon_url',
   'has_community_visible_stats',
   'has_leaderboards',
   'content_descriptorids'],
  'output': 'data/raw/user_games.parquet',
  'size_mb': 512.48,
  'status': 'downloaded'},
 {'config': 'games',
  'rows': 47811,
  'columns': 29,
  'column_names': ['appid',
   'type',
   'name',
   'is_free',
   'required_age',
   'short_description',
   'detailed_description',
   'about_the_game',
   'supported_languages',
   'developers',
   'publishers',
   'platforms',
   'categories',
   'genres',
   'pc_requirements',
   'mac_requirements',
   'linux_requirements',
   'price_currency',
   'price_initial',
   'price_final',
   'price_discount_percent',
   'price_final_formatted',
   'metacritic_score',
   'metacritic_url',
   'recommendations_total',
   'release_date',
   'release_coming_soon',
   'content_descriptor_id

## 5. Create a manifest

The manifest records the dataset source, download time, row counts, column names, and local output paths. This is useful for reproducibility and the Check-in 1 dataset documentation.

In [6]:
manifest = {
    "dataset_id": DATASET_ID,
    "source_url": f"https://huggingface.co/datasets/{DATASET_ID}",
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "configs": results,
}

manifest_path = RAW_DATA_DIR / "manifest.json"

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print("Manifest written to:", manifest_path)

Manifest written to: /Users/robertgeorge/Desktop/DSAN Files/6600/dsan6600-finalproject/data/raw/manifest.json


## 6. Confirm downloaded files

In [7]:
for path in sorted(RAW_DATA_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path.name:25s} {size_mb:10.2f} MB")


games.parquet                 133.21 MB
manifest.json                   0.00 MB
user_games.parquet            512.48 MB
